# 🚗 Silver Order Transformation

## Bronze Source
`automobile_repair.bronze.order` - 19 columns

## Data Quality Analysis (Jan 2026 Sample)
* **Total Orders**: 5,878 orders
* **Order Status**: 73% COMPLETED (4,316) | 17% IN_PROGRESS (977) | 10% OPEN (585)
* **Service Types**: MECHANICAL (26%), INSPECTION (25%), PAINT (25%), BODY (24%)
* **Data Quality Issues**:
  * 243 NULL `technician_id` (~4%) across all statuses - **KEEP WITH FLAG**
  * NULL datetime fields for IN_PROGRESS/OPEN orders - **EXPECTED** (events haven't occurred)
* **No Duplicates**: 1 record per order_id ✅

## Silver Transformation
**16 Essential Columns** for KPIs:
* Core IDs: order_id, store_id, technician_id, technician_name
* Service attributes: service_type, order_status
* Key datetime fields: vehicle_in, vehicle_out, work_start, completion, promised_delivery, actual_delivery
* Time dimensions: order_year, order_month
* **Calculated metrics**: days_in_shop, delivery_accuracy_days (negative = early, positive = late)

**Removed**: customer info (name, phone), vehicle details (no, make, model), planned datetime fields (only keeping actual)

In [0]:
%sql
CREATE OR REPLACE TABLE automobile_repair.silver.order AS
SELECT 
  -- Core identifiers
  order_id,
  store_id,
  technician_id,
  technician_name,
  service_type,
  order_status,
  
  -- Key datetime fields (actual only, not planned)
  vehicle_in_datetime,
  vehicle_out_datetime,
  actual_work_start_datetime,
  actual_completion_datetime,
  promised_delivery_datetime,
  actual_delivery_datetime,
  
  -- Time dimensions for filtering
  YEAR(vehicle_in_datetime) AS order_year,
  MONTH(vehicle_in_datetime) AS order_month,
  
  -- Calculated metrics
  CASE 
    WHEN vehicle_out_datetime IS NOT NULL 
    THEN DATEDIFF(DAY, vehicle_in_datetime, vehicle_out_datetime)
    ELSE NULL 
  END AS days_in_shop,
  
  CASE 
    WHEN actual_delivery_datetime IS NOT NULL AND promised_delivery_datetime IS NOT NULL
    THEN DATEDIFF(DAY, promised_delivery_datetime, actual_delivery_datetime)
    ELSE NULL 
  END AS delivery_accuracy_days
  
FROM automobile_repair.bronze.order
WHERE order_id IS NOT NULL;

In [0]:
%sql
SELECT * FROM automobile_repair.silver.order 
ORDER BY vehicle_in_datetime DESC 
LIMIT 20;

In [0]:
%sql
SELECT 
  service_type,
  order_status,
  COUNT(*) as order_count,
  AVG(days_in_shop) as avg_days_in_shop,
  AVG(delivery_accuracy_days) as avg_delivery_accuracy
FROM automobile_repair.silver.order
WHERE order_year = 2026 AND order_month = 1
GROUP BY service_type, order_status
ORDER BY service_type, order_status;